In [32]:
################### IMPORT THE REQUIRED MODULES

import numpy as np
import neal
import math
import itertools
import sympy as sp
import time

In [33]:
################## CHOOSE THE DWAVE SIMULATED ANNEALING SAMPLER

sampler = neal.SimulatedAnnealingSampler()

In [34]:
################## MAP SPINS {-1,1} TO QUBITS {0,1}

def taufromsig(sig):
    return (sig+1)/2


################### CONVERT FROM BINARY ------> E.G. FROM TAU = [0,1,0] IT RETURNS 0*2^0 + 1*2^1 + 0*2^3 = 2

def Xfromtau(tau): 
    num=0
    for l in range(len(tau)):
        num += tau[l]*2**l
    return int(num)

In [35]:
################## THIS IS TO AUTOMATISE THE REDUCTION OPTIMISING THE NUMBER OF AUXILIARY SPINS. 

################## YOU DO NOT WANT TO TOUCH THIS BOX


def count_couplings(*clists):
    
    occ_list = np.zeros(shape=(0,3))
    
    for l in range(len(clists)):
        
        ncol = clists[l].shape[1]
    
        for i in range(len(clists[l])):
            
            
            minus1 = np.where(clists[l][i][0:ncol-1] == -1)[0]
        
            spins = []
            
            
            if len(minus1) == 0: 
                
                spins = clists[l][i][0:ncol-1]
            
            elif minus1[0] == 2: continue 
        
            else:
                
                
                for j in range(minus1[0]): spins.append(clists[l][i][j])
                    
            
            pairs = list(itertools.combinations(spins, 2)) 
            
            
            for j in range(len(pairs)):
        
                occ = np.where((occ_list[:,0] == pairs[j][0]) & (occ_list[:,1] == pairs[j][1]))[0]

                if len(occ) == 1: occ_list[occ[0]][2] += 1
            
                else: occ_list = np.append(occ_list, np.array([[pairs[j][0], pairs[j][1], 1]]),axis=0)
                    
        
    return occ_list

def replace_pairs(num_spin,most,*higher_coup):
    
    for l in range(len(higher_coup)):
    
        for i in range(len(higher_coup[l])):
            
            ncol = higher_coup[l].shape[1]
            
            minus1 = np.where(higher_coup[l][i][0:ncol-1] == -1)[0]
        
            inds = []
        
            if len(minus1) == 0: 
                
                p_minus1 = ncol-2
                for q in range(ncol-1): inds.append(q)
                

            
            else:
                
                p_minus1 = minus1[0]-1
                for j in range(minus1[0]): inds.append(j)
                    
                
               
            pairs = list(itertools.combinations(inds, 2)) 
            
            
            for j in range(len(pairs)):
                
                i1 = int(pairs[j][0])
                i2 = int(pairs[j][1])
                
    

                if higher_coup[l][i][i1] == most[0] and higher_coup[l][i][i2] == most[1]:
                    
                    higher_coup[l][i][i1] = num_spin
                    higher_coup[l][i][i2] = higher_coup[l][i][p_minus1]
                    higher_coup[l][i][p_minus1] = -1
                    
                    higher_coup[l][i][0:p_minus1].sort()
                    
                    
    return higher_coup

def make_quadratic(num_spin,L,quadratic,linear,*higher_coup):
    
    aux_spin = num_spin-1
    
    while True:
        
        occ_list = count_couplings(*higher_coup)
        
        if len(occ_list) == 0: break
            
        occ_list = occ_list[np.argsort(occ_list[:,2])]
        most = np.array([occ_list[len(occ_list)-1][0],occ_list[len(occ_list)-1][1]])
    
        print("{",int(most[0]),",",int(most[1]),"}")
        aux_spin += 1 
        
        quadratic = np.append(quadratic, np.array([[aux_spin, most[0],-2*L]]),axis=0)
        quadratic = np.append(quadratic, np.array([[aux_spin, most[1],-2*L]]),axis=0)
        linear = np.append(linear, np.array([[aux_spin,3*L]]),axis=0)
            
        occ = np.where((quadratic[:,0] == most[0]) & (quadratic[:,1] == most[1]))[0]
            
        if len(occ) > 0: 
            linear[len(linear)-1][1] += quadratic[occ[0]][2]
            quadratic[occ[0]][2] = L
        
        else:
            quadratic = np.append(quadratic, np.array([[most[0], most[1],L]]),axis=0)
        
        higher_coup = replace_pairs(aux_spin,most,*higher_coup)
    
        occ_list = np.zeros(shape=(0,3))
        
    
    for l in range(len(higher_coup)):
        
        ncol = higher_coup[l].shape[1]
        
        for i in range(len(higher_coup[l])):
            
            if higher_coup[l][i][1] == -1:
                
                occ = np.where(linear[:,0] == higher_coup[l][i][0])[0]
                
                if len(occ) == 0:
                    linear = np.append(linear, np.array([[higher_coup[l][i][0], higher_coup[l][i][ncol-1]]]),axis=0)
                    
                else:
                    linear[occ[0]][1] += higher_coup[l][i][ncol-1]
            else:    
                
                occ = np.where((quadratic[:,0] == higher_coup[l][i][0]) & (quadratic[:,1] == higher_coup[l][i][1]))[0]
                occ1 = np.where((quadratic[:,1] == higher_coup[l][i][0]) & (quadratic[:,0] == higher_coup[l][i][1]))[0]
                
                if len(occ) == 0 and len(occ1) == 0:
                    quadratic = np.append(quadratic, np.array([[higher_coup[l][i][0],higher_coup[l][i][1], higher_coup[l][i][ncol-1]]]),axis=0)
    
                elif len(occ) != 0:
                    quadratic[occ[0]][2] += higher_coup[l][i][ncol-1]
                
                elif len(occ1) != 0:
                    quadratic[occ1[0]][2] += higher_coup[l][i][ncol-1]
    
    print("Number of auxiliary spins: ",aux_spin + 1 - n_var*beta)
    
    
    return quadratic, linear, aux_spin 

In [36]:
################## TEST IF A PROPOSED SOLUTION COMING FROM THE ANNEALER IS A REAL SOLUTION

def itisasolution(lambdas):
    
    z = lambdas[0:n_var]

    return  (z[0] + z[1] - 6)**2

In [37]:
beta = 4      ################## NUMBER OF QUBITS FOR EACH VARIABLE

n_var = 2     ################## NUMBER OF VARIABLES

shift = 0     ################## EVENTUAL SHIFT IN THE BINARY REPRESENTATION

max_exp = 2   ################## THE LARGEST EXPONENT THAT APPEARS IN THE HAMILTONIAN

In [38]:
################## EQS IS A LIST WHICH CONTAINS THE EQUATION(S) WE WANT TO SOLVE IN TERM OF THE SYMPY SYMBOLS Z

eqs = []

z = sp.symarray("z",n_var)

eq = (z[0] + z[1] - 6)**2    ###### HERE IS THE EQUATION/PROBLEM YOU WANT TO SOLVE


eqs.append(eq) 

In [39]:
################## THIS IS JUST TO CHOOSE, IF NEEDED, A SUBSET OF THE EQUATIONS/VARIABLES INVOLVED.

################## YOU CAN INGORE IT

eq_chosen = []
var_chosen = []

n_eq = len(eqs)

var_chosen= [n for n in range(n_var)]
eq_chosen = [n for n in range(n_eq)]

In [40]:
################## BINARY ENCODING

sym = sp.symarray("x", n_var*beta)                       
sigmas = np.array([2**n for n in range(beta)])

In [48]:
################## variables IS A LIST CONTAINING THE BINARY REPRESENTATION OF z[0] and z[1]

variables = []



for i in range(n_var): 
    variables.append(np.tensordot(sigmas, sym[i*beta:(i+1)*beta], axes=(0,0)) + shift) 
    
print(variables)

[x_0 + 2*x_1 + 4*x_2 + 8*x_3, x_4 + 2*x_5 + 4*x_6 + 8*x_7]


In [49]:
################## CONSTRUCT THE HAMILTONIAN AS A SYMPY POLYNOMIAL 


sq = 0
for i in range(n_eq): sq += eqs[eq_chosen[i]]
print("The Hamiltonian is: \n\n",sq)
squares = sq.copy()


for i in range(n_var): sq = sq.subs(z[i],variables[i])  ############ IN THE HAMILTONIAN, WE SUBSTITUTE Z WITH ITS BINARY REPRESENTATION
    
start = time.time()
H = sp.poly((sq),*sym)
print("\n\nHamiltonian written in terms of taus:", time.time()-start)

The Hamiltonian is: 

 (z_0 + z_1 - 6)**2


Hamiltonian written in terms of taus: 0.016132831573486328


In [50]:
##################### REPLACE TAU^N WITH TAU AS TAU = {0,1}

print("\nReplace powers...")



lsym = [n for n in range(n_var*beta)]
mapping = {sym[i]**j: sym[i] for i,j in list(itertools.product(lsym, [n for n in range(max_exp+1)]))}

start = time.time()
H = H.xreplace(mapping)
print("Done in", time.time()-start)
        
H = sp.poly(H,*sym)

print(H)


Replace powers...
Done in 0.004362821578979492
Poly(4*x_0*x_1 + 8*x_0*x_2 + 16*x_0*x_3 + 2*x_0*x_4 + 4*x_0*x_5 + 8*x_0*x_6 + 16*x_0*x_7 - 11*x_0 + 16*x_1*x_2 + 32*x_1*x_3 + 4*x_1*x_4 + 8*x_1*x_5 + 16*x_1*x_6 + 32*x_1*x_7 - 20*x_1 + 64*x_2*x_3 + 8*x_2*x_4 + 16*x_2*x_5 + 32*x_2*x_6 + 64*x_2*x_7 - 32*x_2 + 16*x_3*x_4 + 32*x_3*x_5 + 64*x_3*x_6 + 128*x_3*x_7 - 32*x_3 + 4*x_4*x_5 + 8*x_4*x_6 + 16*x_4*x_7 - 11*x_4 + 16*x_5*x_6 + 32*x_5*x_7 - 20*x_5 + 64*x_6*x_7 - 32*x_6 - 32*x_7 + 36, x_0, x_1, x_2, x_3, x_4, x_5, x_6, x_7, domain='ZZ')


In [42]:
################## EXTRACT THE COUPLINGS AS A DICTIONARY, YOU DO NOT NEED TO EDIT THIS

dict_coup = {}

coup_list = list(H.as_dict().items())

for i in range(len(coup_list)):
    
    spins = np.where(np.array(coup_list[i][0][:]) == 1)[0]
    
    if len(spins) > 0:
        
        spins = np.append(spins,coup_list[i][1])
        
        if len(spins)-1 in dict_coup:
            
            dict_coup[len(spins)-1] = np.append(dict_coup[len(spins)-1],np.array([spins]),axis=0)
             
        else:
            
            dict_coup[len(spins)-1] = np.zeros(shape = (0,len(spins)))
            dict_coup[len(spins)-1] = np.append(dict_coup[len(spins)-1],np.array([spins]),axis=0)
            
quad = dict_coup[2]
lin = dict_coup[1]



del dict_coup[1]
del dict_coup[2]


In [43]:
################## AUTOMATIC REDUCTION OF THE HAMILTONIAN

L = 2 * int(max(list(H.as_dict().values())))

reduced = make_quadratic(n_var*beta,L,quad,lin,*list(dict_coup.values()))

Number of auxiliary spins:  0


In [44]:
################## CONSTRUCT h AND J

quad = reduced[0].copy()
lin = reduced[1].copy()
aux_spin = reduced[2]

J = np.zeros(shape=(aux_spin+1,aux_spin+1))
h = np.zeros(shape=(aux_spin+1))
   
    
for i in range(len(quad)):
    
    qi0 = int(quad[i][0])
    qi1 = int(quad[i][1])
    qi2 = quad[i][2]
        
    J[qi0][qi1] = qi2/4
    
    
        
    h[qi0] += qi2/4
    h[qi1] += qi2/4
        
for i in range(len(lin)):
    
    li0 = int(lin[i][0])
    li1 = lin[i][1]
        
    h[li0] += li1/2

In [45]:
nreads = 2000           # NUMBER OF ANNEAL RUNS     

answer = sampler.sample_ising(h, J, num_reads=nreads)         # SIMULATED ANNEALING

print(answer)           # PRINT THE OUTPUT


      0  1  2  3  4  5  6  7 energy num_oc.
0    -1 +1 -1 -1 -1 -1 +1 -1 -123.5       1
1    -1 +1 +1 -1 -1 -1 -1 -1 -123.5       1
2    +1 -1 -1 -1 +1 -1 +1 -1 -123.5       1
3    +1 -1 +1 -1 +1 -1 -1 -1 -123.5       1
4    -1 +1 +1 -1 -1 -1 -1 -1 -123.5       1
5    +1 -1 -1 -1 +1 -1 +1 -1 -123.5       1
6    -1 +1 +1 -1 -1 -1 -1 -1 -123.5       1
7    +1 -1 +1 -1 +1 -1 -1 -1 -123.5       1
8    +1 -1 -1 -1 +1 -1 +1 -1 -123.5       1
9    +1 -1 -1 -1 +1 -1 +1 -1 -123.5       1
10   +1 -1 +1 -1 +1 -1 -1 -1 -123.5       1
11   -1 +1 +1 -1 -1 -1 -1 -1 -123.5       1
12   -1 +1 +1 -1 -1 -1 -1 -1 -123.5       1
13   -1 +1 +1 -1 -1 -1 -1 -1 -123.5       1
14   -1 +1 +1 -1 -1 -1 -1 -1 -123.5       1
15   -1 -1 -1 -1 -1 +1 +1 -1 -123.5       1
16   +1 +1 -1 -1 +1 +1 -1 -1 -123.5       1
17   +1 -1 -1 -1 +1 -1 +1 -1 -123.5       1
18   -1 -1 -1 -1 -1 +1 +1 -1 -123.5       1
19   -1 -1 +1 -1 -1 +1 -1 -1 -123.5       1
20   +1 +1 -1 -1 +1 +1 -1 -1 -123.5       1
22   +1 -1 +1 -1 +1 -1 -1 -1 -12

In [46]:
################### THIS IS JUST TO CONVERT FROM SPIN VALUES TO NUMBERS


Lambdasig = []
Lambda = []

for i in range(n_var): 
    
    Lambdasig.append(np.zeros(shape=(nreads,beta)))
    Lambda.append(np.zeros(shape=(nreads)))

values = np.zeros(shape=(nreads))

for i in range(nreads):
    for j in range(beta):
        for k in range(n_var):
        
            Lambdasig[k][i][j] = answer.record[i][0][j+k*beta]
        
    for k in range(n_var):

        Lambda[k][i] = Xfromtau(taufromsig(Lambdasig[k][i])) + shift
        
    current_Lambda = list(zip(*Lambda))[i]
    
    values[i] = itisasolution(current_Lambda)
    
    
    if values[i] == 0:  # IF IT IS A SOLUTION
        if answer.record[i][1] == answer.first.energy: # AND IF IT IS ACTUALLY THE MINIMUM STATE FOUND
            print("\n\n\nSolution:",i,current_Lambda) # THEN ---> PRINT THE NUMBERS 
                
         




Solution: 0 (2.0, 4.0)



Solution: 1 (6.0, 0.0)



Solution: 2 (1.0, 5.0)



Solution: 3 (5.0, 1.0)



Solution: 4 (6.0, 0.0)



Solution: 5 (1.0, 5.0)



Solution: 6 (6.0, 0.0)



Solution: 7 (5.0, 1.0)



Solution: 8 (1.0, 5.0)



Solution: 9 (1.0, 5.0)



Solution: 10 (5.0, 1.0)



Solution: 11 (6.0, 0.0)



Solution: 12 (6.0, 0.0)



Solution: 13 (6.0, 0.0)



Solution: 14 (6.0, 0.0)



Solution: 15 (0.0, 6.0)



Solution: 16 (3.0, 3.0)



Solution: 17 (1.0, 5.0)



Solution: 18 (0.0, 6.0)



Solution: 19 (4.0, 2.0)



Solution: 20 (3.0, 3.0)



Solution: 22 (5.0, 1.0)



Solution: 23 (6.0, 0.0)



Solution: 24 (6.0, 0.0)



Solution: 25 (0.0, 6.0)



Solution: 26 (0.0, 6.0)



Solution: 27 (3.0, 3.0)



Solution: 28 (1.0, 5.0)



Solution: 29 (6.0, 0.0)



Solution: 30 (0.0, 6.0)



Solution: 31 (0.0, 6.0)



Solution: 32 (2.0, 4.0)



Solution: 33 (6.0, 0.0)



Solution: 34 (3.0, 3.0)



Solution: 35 (2.0, 4.0)



Solution: 36 (4.0, 2.0)



Solution: 37 (2.0, 4.0)



Solution

Solution: 442 (4.0, 2.0)



Solution: 443 (6.0, 0.0)



Solution: 444 (2.0, 4.0)



Solution: 445 (1.0, 5.0)



Solution: 446 (5.0, 1.0)



Solution: 447 (1.0, 5.0)



Solution: 448 (2.0, 4.0)



Solution: 449 (3.0, 3.0)



Solution: 450 (3.0, 3.0)



Solution: 451 (5.0, 1.0)



Solution: 452 (6.0, 0.0)



Solution: 454 (6.0, 0.0)



Solution: 455 (0.0, 6.0)



Solution: 456 (2.0, 4.0)



Solution: 457 (2.0, 4.0)



Solution: 458 (1.0, 5.0)



Solution: 459 (3.0, 3.0)



Solution: 460 (4.0, 2.0)



Solution: 461 (1.0, 5.0)



Solution: 462 (0.0, 6.0)



Solution: 463 (1.0, 5.0)



Solution: 464 (5.0, 1.0)



Solution: 465 (4.0, 2.0)



Solution: 466 (1.0, 5.0)



Solution: 467 (1.0, 5.0)



Solution: 468 (4.0, 2.0)



Solution: 469 (2.0, 4.0)



Solution: 470 (1.0, 5.0)



Solution: 471 (4.0, 2.0)



Solution: 472 (4.0, 2.0)



Solution: 473 (0.0, 6.0)



Solution: 474 (4.0, 2.0)



Solution: 475 (6.0, 0.0)



Solution: 476 (6.0, 0.0)



Solution: 477 (3.0, 3.0)



Solution: 478 (2.0, 

Solution: 838 (4.0, 2.0)



Solution: 839 (1.0, 5.0)



Solution: 840 (1.0, 5.0)



Solution: 841 (2.0, 4.0)



Solution: 842 (2.0, 4.0)



Solution: 843 (2.0, 4.0)



Solution: 844 (6.0, 0.0)



Solution: 845 (5.0, 1.0)



Solution: 846 (2.0, 4.0)



Solution: 847 (2.0, 4.0)



Solution: 848 (2.0, 4.0)



Solution: 849 (0.0, 6.0)



Solution: 850 (6.0, 0.0)



Solution: 851 (6.0, 0.0)



Solution: 852 (2.0, 4.0)



Solution: 853 (5.0, 1.0)



Solution: 854 (1.0, 5.0)



Solution: 855 (5.0, 1.0)



Solution: 856 (5.0, 1.0)



Solution: 857 (3.0, 3.0)



Solution: 858 (6.0, 0.0)



Solution: 859 (3.0, 3.0)



Solution: 860 (4.0, 2.0)



Solution: 861 (1.0, 5.0)



Solution: 862 (2.0, 4.0)



Solution: 863 (4.0, 2.0)



Solution: 864 (3.0, 3.0)



Solution: 865 (3.0, 3.0)



Solution: 866 (6.0, 0.0)



Solution: 867 (4.0, 2.0)



Solution: 868 (6.0, 0.0)



Solution: 869 (1.0, 5.0)



Solution: 870 (2.0, 4.0)



Solution: 872 (6.0, 0.0)



Solution: 874 (1.0, 5.0)



Solution: 875 (6.0, 

Solution: 1261 (5.0, 1.0)



Solution: 1262 (5.0, 1.0)



Solution: 1263 (3.0, 3.0)



Solution: 1264 (5.0, 1.0)



Solution: 1265 (5.0, 1.0)



Solution: 1266 (5.0, 1.0)



Solution: 1267 (1.0, 5.0)



Solution: 1268 (0.0, 6.0)



Solution: 1269 (1.0, 5.0)



Solution: 1270 (2.0, 4.0)



Solution: 1271 (6.0, 0.0)



Solution: 1272 (4.0, 2.0)



Solution: 1273 (0.0, 6.0)



Solution: 1274 (4.0, 2.0)



Solution: 1275 (6.0, 0.0)



Solution: 1276 (4.0, 2.0)



Solution: 1277 (2.0, 4.0)



Solution: 1278 (2.0, 4.0)



Solution: 1279 (5.0, 1.0)



Solution: 1280 (6.0, 0.0)



Solution: 1281 (1.0, 5.0)



Solution: 1282 (6.0, 0.0)



Solution: 1283 (2.0, 4.0)



Solution: 1284 (3.0, 3.0)



Solution: 1285 (2.0, 4.0)



Solution: 1286 (6.0, 0.0)



Solution: 1287 (3.0, 3.0)



Solution: 1288 (2.0, 4.0)



Solution: 1289 (5.0, 1.0)



Solution: 1290 (0.0, 6.0)



Solution: 1291 (4.0, 2.0)



Solution: 1292 (0.0, 6.0)



Solution: 1293 (2.0, 4.0)



Solution: 1294 (2.0, 4.0)



Solution: 1295




Solution: 1654 (3.0, 3.0)



Solution: 1655 (3.0, 3.0)



Solution: 1656 (1.0, 5.0)



Solution: 1658 (4.0, 2.0)



Solution: 1659 (5.0, 1.0)



Solution: 1660 (2.0, 4.0)



Solution: 1661 (4.0, 2.0)



Solution: 1662 (6.0, 0.0)



Solution: 1663 (2.0, 4.0)



Solution: 1664 (2.0, 4.0)



Solution: 1665 (2.0, 4.0)



Solution: 1666 (1.0, 5.0)



Solution: 1667 (2.0, 4.0)



Solution: 1668 (2.0, 4.0)



Solution: 1669 (2.0, 4.0)



Solution: 1670 (4.0, 2.0)



Solution: 1671 (2.0, 4.0)



Solution: 1672 (5.0, 1.0)



Solution: 1673 (5.0, 1.0)



Solution: 1674 (1.0, 5.0)



Solution: 1675 (6.0, 0.0)



Solution: 1676 (6.0, 0.0)



Solution: 1677 (2.0, 4.0)



Solution: 1678 (3.0, 3.0)



Solution: 1680 (6.0, 0.0)



Solution: 1681 (0.0, 6.0)



Solution: 1682 (5.0, 1.0)



Solution: 1683 (6.0, 0.0)



Solution: 1684 (3.0, 3.0)



Solution: 1685 (3.0, 3.0)



Solution: 1686 (3.0, 3.0)



Solution: 1687 (5.0, 1.0)



Solution: 1688 (6.0, 0.0)



Solution: 1689 (2.0, 4.0)



Solution: 1